# ClinIQ — Stage C: Teacher SFT Training

Fine-tunes Qwen3.5-9B on clinical triage data using Unsloth + LoRA.

In [ ]:
# @title ## 🔐 Load Secrets
import os
from google.colab import userdata

for key in ["HF_TOKEN", "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "S3_BUCKET_NAME"]:
    os.environ[key] = userdata.get(key)

os.environ["AWS_DEFAULT_REGION"] = "us-east-1"
os.environ["HF_REPO_ID"] = "Zolisa/cliniq-0.5b"
os.environ["NTFY_TOPIC"] = "cliniq-pipeline"
os.environ["TEACHER_MODEL"] = "Qwen/Qwen3.5-9B"
os.environ["STUDENT_MODEL"] = "Qwen/Qwen2.5-0.5B-Instruct"
os.environ["TEACHER_ADAPTER"] = "null"
print("✅ Secrets loaded")

In [ ]:
# @title ## 📦 Install Dependencies
!pip install -q unsloth trl transformers torch datasets accelerate boto3 huggingface-hub python-dotenv requests rouge-score bitsandbytes

In [ ]:
# @title ## 📥 Clone Repo & Prepare Data
import os

if not os.path.exists('/content/cliniq'):
    !git clone https://github.com/ZolisaSilolo/Knowldge-Distillation103.git /content/cliniq

%cd /content/cliniq
%env PYTHONPATH=/content/cliniq

!python data/fetch_regulations.py
!python data/prepare.py

In [ ]:
# @title ## 🚀 Run Teacher SFT Training
import sys
sys.path.insert(0, '/content/cliniq')

from unsloth import FastLanguageModel
import torch
from transformers import TrainingArguments
from trl import SFTTrainer
from data.dataset import load_sft_dataset
from utils.checkpoint import upload_checkpoint, upload_metrics

MODEL_NAME = "Qwen/Qwen3.5-9B"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

train_dataset = load_sft_dataset("data/processed/train.jsonl", tokenizer, MAX_SEQ_LENGTH)
eval_dataset  = load_sft_dataset("data/processed/eval.jsonl",  tokenizer, MAX_SEQ_LENGTH)

training_args = TrainingArguments(
    output_dir="./outputs/stage_c",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=3,
    fp16=True,
    logging_steps=10,
    save_steps=50,
    eval_strategy="steps",
    eval_steps=50,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
    packing=False,
)

train_result = trainer.train()
eval_results = trainer.evaluate()
print(f"Train loss: {train_result.training_loss:.4f} | Eval loss: {eval_results['eval_loss']:.4f}")

In [ ]:
# @title ## 💾 Save Adapter & Upload to S3 + HF
from pathlib import Path
from utils.checkpoint import upload_checkpoint, upload_metrics

final_dir = Path("outputs/stage_c/final_adapter")
final_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(final_dir))
tokenizer.save_pretrained(str(final_dir))
print(f"💾 Adapter saved to {final_dir}")

# Upload to S3 (triggers EventBridge → ntfy notification)
upload_checkpoint(str(final_dir), "stage_c", s3_prefix="checkpoints", cleanup=True)
upload_metrics({
    "stage": "stage_c",
    "eval_loss": eval_results["eval_loss"],
    "train_loss": train_result.training_loss,
    "epochs": 3,
}, "stage_c")
print("✅ Stage C complete — checkpoint + metrics uploaded to S3")